# 生成对抗网络实践

生成对抗网络（Generative Adversarial Nets，以下简称GAN）是Ian J. Goodfellow等人在2014年的论文[Generative Adversarial Nets](https://arxiv.org/pdf/1406.2661.pdf)中提出的一种评估生成模型的方法。GAN中巧妙的提出了以对抗的方法来训练生成和判别两个模型，在两者的对抗过程中不断改进，最终达到一种平衡。神经网络奠基人之一的Yann LeCun称赞对抗训练是“在过去10年中最有趣的机器学习想法”。


## 生成算法与判别算法

在机器学习中，我们主要处理的数据为特征集$X$和对应的标签$y$。判别方法对$X \to y$这种特征到标签的映射关系进行建模，也就是说，给定特征$X$，求解标签$y$的条件分布。

而生成方法则恰好相反，它对$y \to X$，即对标签到特征的关系进行建模。给定一个标签$y$，生成算法尝试预测(生成)与之对应的特征。

GAN的巧妙之处就在于，引入一个生成模型G和一个判别模型D。G用来捕捉数据的分布，并生成伪造的数据，而D则用来判定一条数据是由G伪造的，还是真的来源于真实数据集。在训练中，G的目标是尽最大可能让D犯错误，而D的目标则是尽可能辨别出伪造的数据。最终两者达到一种平衡：G可以重现数据的特征，而D对数据来源的判断都是50%的概率。


## minmax和maxmin二元博弈

minmax是"minimize max"的简称，即：使某种最大值尽量小，是一种广泛用于人工智能、博弈论中的决策方法，在最坏情况下使最大损失(max loss)最小化(minimize)。与之相反的是maximize min，即将最小收益(min gain)最大化(maximize)。下面，我们就对这两种策略进行说明（为了便于理解，以下我们将minmax写为mini-max）。

**maxi-min策略**
在博弈中，对于一个玩家而言maxi-min value是他在不知道其他玩家策略的情况下，确保能够取得的最大收益，换言之，也是其他玩家在知道该玩家策略下，能迫使该玩家接受的最小值。第一种表述方法更易于我们按照字面的maxi-min来理解其含义，即：我已经知道所有情况下我能获得的最小收益，我要从中取得最大的值。我们用下面的公式表示maxi-min收益：

$$\underline{v_i} = \max_{a_i}\min_{a_{-i}}v_i(a_i, a_{-i}) $$

其中：

- $i$表示当前玩家，$-i$表示其他玩家，$a_i$表示当前玩家的策略，$a_{-i}$表示其他玩家的策略

- $v_i$表示玩家$i$的收益函数

一个玩家的maxi-min收益即在最坏的情况下（收益最小），尽可能取较大值。假设两个玩家R和C，玩家R有3个策略：T，M，B，玩家C有2个策略L、R，两者每种策略组合下的最大收益和最小收益如下表所示：


| NA |L|R|
|:--|:--:|--:|
|T|3,1|2,-20|
|M|5,0|-10,1|
|B|-100,2|4,4|

其中第一数字代表玩家R的收益，第二个数字代表玩家C的收益。对于玩家R来说，三个策略的收益分别是：

- $ v_T = [3, 2]，最小为2 $
- $ v_M = [5, -10]，最小为-10 $
- $ v_B = [-100, 4]，最小为-100 $

因此，在maxi-min策略下，玩家R应该采取的策略为T，收益为$\underline{v_R} = 2$。同理玩家R的两个策略的收益分别是：

- $ v_L = [1, 0, 2]，最小为0 $
- $ v_R = [-20, 1, 4]，最小为-20 $

因此，在maxi-min策略下，玩家R应采取的策略为L，收益为$\underline{v_L} = 0$。


**mini-max策略**

在mini-max策略下，一个玩家的mini-max收益为：其他玩家不知道该玩家策略的前提下，可以迫使该玩家可以接受的最小收益；等价于：在知道其他玩家策略的前提下，该玩家可以确保获得的最大收益。我们用第一种表述更易于理解mini-max的含义，即：其他玩家不知道该玩家的策略，那就在该玩家能取得的最大收益(max value)中，迫使该玩家取最小的(minimize)。mini-max形式化的表述为：

$$\bar{v_i} = \min_{a_{-i}}\max_{a_i}v_i(a_i, a_{-i}) $$


在玩家R和C的博弈中，对于玩家R：

- 如果玩家C采取策略L，玩家R的收益为$v = [3, 5, -100]$，最大为5
- 如果玩家C采取策略R，玩家R的收益为$v = [2, -10, 4]$，最大为4

因此，如果玩家C知道玩家R的策略，则C可以迫使R接受最大收益为4，即$\bar{v_R} = 4$。

而对于玩家C：

- 如果玩家R采取策略T，玩家C的收益为$v = [1, -20]$，最大值为1
- 如果玩家R采取策略M，玩家C的收益为$v = [0, 1]$，最大值为1
- 如果玩家R采取策略B，玩家C的收益为$v = [2, 4]$，最大值为2

如果玩家R知道玩家C的策略，则可以迫使C取最大收益为1，即$\bar{v_C} = 1$


**GAN中的mini-max二元博弈**

在GAN中，两个玩家为生成模型G和判别模型D，两者进行mini-max二元博弈的value function定义为：

$$\min_{G}\max_{D}V(D, G) = \mathbb{E}_{x\sim{p_{data}(x)}}[log D(x)] + \mathbb{E}_{z\sim{p_z(Z)}}[log(1-D(G(Z)))]$$

直观的理解，就是G模型在知道D模型的策略时，迫使D的最大收益尽可能的小。我们将这个mini-max二元博弈过程逐步进行分析，首先，博弈过程可以理解为：


$$\min_{G}(\max_{D}V(D, G))$$

也即D尽量使收益函数取较大值，而G则迫使D的最大收益尽量小，在收益函数的表达式中：

$$ \mathbb{E}_{x\sim{p_{data}(x)}}[log D(x)] + \mathbb{E}_{z\sim{p_z(Z)}}[log(1-D(G(Z)))]$$

判别模型D输出结果$D(x)$为一个$sigmoid$函数的输出结果，范围为0到1，接近0代表判定数据不是来自于真实数据集（由生成模型G伪造），接近1代表判定数据来自于真实数据集。

对于判别模型D，在给定的真实数据分布$x\sim{P_{data}(x)}$下，确保$D(x) \to{1}$，以保证$log D(x)$取最大值。同时，对于给定的噪声数据分布$z\sim{P_z(Z)}$，$G(Z)$为生成模型伪造的数据，因此$D(G(Z))$为判别模型对伪造数据的判定结果，该结果应该趋近于0，则$1-D(G(Z)) \to 1$，保证$log(1 - D(G(Z)))$取最大值。因此，对于真实数据和伪造数据的判定值的和也能够取最大值。

这样，我们定义的结果函数$\mathbb{E}_{x\sim{p_{data}(x)}}[log D(x)] + \mathbb{E}_{z\sim{p_z(Z)}}[log(1-D(G(Z)))$恰好符合我们对问题的预期：对于真实数据，判别模型输出$D(x)=1$，对于伪造数据，判别模型输出$D(G(Z))=0$，这样，确保判别模型在mini-max二元博弈中取最大值。而对于生成模型G，则是强迫判别模型接受在最大值集合中取最小的值。这就是D和G的mini-max二元博弈的数学定义。

按照上述等式的定义，对于生成模型G，要在对抗训练中确保$log(1 - D(G(Z)))$取最小值（D目标为使该值取最大），但是，在刚开始训练时，G从噪声数据中生成的数据表现与真实数据差距较大，判别模型D很容易识别出数据的真伪，该值容易出现饱和。因此，我们训练G使得$logD(G(z))$最大，这样，在G的训练中可以保证更好的梯度。

## 构建Generator和Discriminator

下面，我们分别构建G模型和D模型，并使用MNIST数据集对两个模型进行对抗训练。在原论文中，作者提到两个模型是多层感知机，因此可以使用梯度下降和反向传播进行训练。简单的理解，我们可以认为多层感知机是每个层都带有激活函数的神经网络。


### Generator

对于生成模型G，需要输入一组噪声数据，经过若干层神经元的处理，输出伪造的图片。

In [ ]:
import numpy as np
from keras.models import Sequential, Model
from keras.layers import Dense, LeakyReLU, BatchNormalization, Reshape, Input

latent_dim = 100
mnist_shape = (28, 28, 1)

model = Sequential()

model.add(Dense(256, input_dim=latent_dim))
model.add(LeakyReLU(alpha=0.2))
model.add(BatchNormalization(momentum=0.8))
model.add(Dense(512))
model.add(LeakyReLU(alpha=0.2))
model.add(BatchNormalization(momentum=0.8))
model.add(Dense(1024))
model.add(LeakyReLU(alpha=0.2))
model.add(BatchNormalization(momentum=0.8))
model.add(Dense(np.prod(mnist_shape), activation='tanh'))
model.add(Reshape(mnist_shape))

model.summary()

noise = Input(shape=(latent_dim,))
img = model(noise)

G = Model(noise, img)

### Discriminator

本质上，判别模型D是一个二分类的神经网络模型。输入数据为28x28的图片，输出一个二分类的概率预测，对于二分类，我们直接使用sigmoid函数输出一个单个的值。

In [ ]:
from keras.layers import Flatten

model = Sequential()

model.add(Flatten(input_shape=mnist_shape))
model.add(Dense(512))
model.add(LeakyReLU(alpha=0.2))
model.add(Dense(256))
model.add(LeakyReLU(alpha=0.2))
model.add(Dense(1, activation='sigmoid'))
model.summary()

img = Input(shape=mnist_shape)
validity = model(img)

D = Model(img, validity)

### 准备数据集并训练模型

我们使用MNIST数据集进行训练，将G模型训练成可以从噪声数据生成MNIST手写数据的模型。

In [ ]:
from keras.datasets import mnist

BATCH_SIZE = 64

# 从MNIST加载的数据都是真实数据，我们只用于GAN中的真实数据分布，因此不需要对标签进行处理
(X_train, _), (_, _) = mnist.load_data()

# 对数据进行映射，灰度值[0, 255]映射到[-1, 1]
X_train = X_train / 127.5 - 1.
X_train = np.expand_dims(X_train, axis=3)
print(X_train.shape)

# 为判别模型D构建二分类结果的labels
valid = np.ones((BATCH_SIZE, 1))
fake = np.zeros((BATCH_SIZE, 1))

### 定义二元博弈关系的模型和损失函数

我们构建好G模型和D模型后，将其堆叠起来形成一个新的模型combined，用于描述两者的二元博弈关系。注意我们并不直接对G模型进行训练，而是先用G模型生成伪造图片，去训练D模型。然后在combined模型中，用G模型生成图片，再用训练的D模型判别结果，将结果与真实数据标签进行交叉熵损失函数的训练，以使G模型向生成真实数据的方向训练。

In [ ]:
from keras.optimizers import Adam
optimizer = Adam(0.0002, 0.5)

D.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# 构建combined model
z = Input(shape=(latent_dim,))
img = G(z)
# D模型在compile之后设置trainable为False，这样在combined模型中不会被训练，而在已经编译的D中权重依然会进行训练
D.trainable = False 
validity = D(img)

combined = Model(z, validity)
combined.compile(loss='binary_crossentropy', optimizer=optimizer)

combined.summary()

epochs = 10000
for epoch in range(epochs):
    # 首先训练D模型
    
    # 从MNIST数据集中随机选取一批图片作为真实图片
    idx = np.random.randint(0, X_train.shape[0], BATCH_SIZE)
    imgs = X_train[idx]

    noise = np.random.normal(0, 1, (BATCH_SIZE, latent_dim))

    # 将噪声数据传入G模型，生成伪造的图片
    gen_imgs = G.predict(noise)

    # 使用真伪两组图片训练D模型，判断真实数据和伪造数据
    d_loss_real = D.train_on_batch(imgs, valid)
    d_loss_fake = D.train_on_batch(gen_imgs, fake)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # 训练G模型
    noise = np.random.normal(0, 1, (BATCH_SIZE, latent_dim))

    # 将噪声数据和真实数据标签（全部为1）传入combined模型
    # combined模型中G模型将生成图片，并由D模型输出判别结果，再以传入的valid（全部为1）作为标签进行训练
    # 这样，G模型向着生成真实数据的目标进行训练
    g_loss = combined.train_on_batch(noise, valid)

    if epoch % 1000 == 0:
        print ("%d [D loss: %f, acc.: %.2f%%] [G loss: %f]" % (epoch, d_loss[0], 100*d_loss[1], g_loss))

接下来，我们对训练的生成模型和判别模型进行验证。我们随机生成一批噪点数据，用G模型生成伪造的数据，并用D模型判定数据为真实数据的概率

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline



plt.figure(figsize=(10, 8))
for i in range(8):
    noise = np.random.normal(0, 1, (1, latent_dim))
    gen_imgs = G.predict(noise)
    score = D.predict(gen_imgs)[0][0]
    plt.subplot(2, 4, i+1)
    plt.axis('off')
    plt.imshow(np.squeeze(gen_imgs), cmap='gray')
    plt.title('score: {:.2f}'.format(score))
plt.show()

我们可以看到，G模型生成的手写字体有一些已经比较接近真实数据，而D模型对于这些接近真实数据的伪造数据，预测的概率比较接近0.5。

## GAN的扩展

GAN论文发表后，在业界引起了广泛的研究和探讨，产生了很多基于GAN的扩展模型，例如使用深度卷积网络定义模型的[DCGAN](https://arxiv.org/abs/1511.06434)，基于GAN的图片转换方法[pix2pix](https://arxiv.org/abs/1611.07004)等，并产生了很多有趣的应用。接下来我们以[CycleGAN](https://arxiv.org/abs/1703.10593)为例，介绍GAN的相关应用。

### CycleGAN

CycleGAN是在2017年的论文《Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks
》中提出的，从论文标题可以看出，CycleGAN提出了一种不需要配对的图像转换方法。在之前的相关研究中，一般在训练时需要提供一套匹配的图像数据才能实现图像的转换，而CycleGAN则是直接在两种不同风格的图像数据集中学习特征，然后尝试用GAN的方法训练模型，最终让模型可以把一种数据集中的风格特征套用到另一种数据集中。


在CycleGAN中，我们假设有两套不同风格的图像数据$X$和$Y$（如一组是实拍风景，另一组是油画，两者内容不必匹配），按照GAN的思路，我们可以构建一个生成模型G，对$X\to{Y}$进行建模，然后构建一个判别模型D，判断图片是否是G生成的伪造图像。然而，CycleGAN的作者在实践中发现，$X \to{Y}$的映射是有多种解的，很难保证两者的映射关系在视觉上是有意义的，并且常规的GAN手段也容易引起转换效果的下降（[mode collapse](https://aiden.nibali.org/blog/2017-01-18-mode-collapse-gans/)）：$X$的数据分布被映射到$Y$中的少数几个实例上。

CycleGAN的作者提出，我们不仅保证映射关系$G: X \to{Y}$，还可以引入另一个映射关系$F: Y\to{X}$，在训练时，我们同时训练两种映射$G和F$，并引入一个新的"循环一致性损失"(cycle consistent loss)，即：对映射好的关系，再用反向映射，使两次映射的结果向原输入拟合：

$$F(G(x)) \approx {x},     G(F(y)) \approx {y}$$

这样，综合生成对抗损失和循环一致性损失，我们就可以实现无需配对的图像转换。这种方法具备更好的泛化性，可以用于图片风格转换，物体变换，图片增强等场景中。

### CycleGAN中的生成对抗网络

在CycleGAN的实现中，我们对$X \to {Y}$的映射构建一个生成模型$G$，假设$G(x) = \hat{y}$。再构建一个判别模型$D_y$（因为该判别模型要判断真实的数据$y$和生成的数据$\hat{y}$）。同理，对$Y \to{X}$也构建一个生成模型$F$和判别模型$D_x$。此外，为了保证映射在双向都是有效的，我们再引入循环一致性损失，当数据从一个样本空间映射到另一个空间，再反向映射回来，应该保持与原数据一致，即：

$$x\to {G(x)} \to {F(G(x))} \approx {x}$$

$$y\to {F(y)} \to {G(F(y))} \approx {y}$$

可以看到，$x$经过$G$和$F$映射又回到自身，这就是Cycle(循环)的含义。这些映射关系如下图所示：

![cycle-gan-arch](img/cycle-gan-arch.png)

### CycleGAN中的损失函数

在CycleGAN中，损失函数包含两部分，一部分为两个生成对抗模型$G和D_y$，$F和D_x$的损失：

$$\mathcal{L}_{GAN}(G, D_Y, X, Y) = \mathbb{E}_{y\sim{p_{data}}(y)}[logD_Y(y)] + \mathbb{E}_{x\sim{p_{data}}(x)}[log(1-D_Y(G(x)))]$$

$$\mathcal{L}_{GAN}(F, D_X, Y, X) = \mathbb{E}_{x\sim{p_{data}}(x)}[logD_X(x)] + \mathbb{E}_{y\sim{p_{data}}(y)}[log(1-D_X(F(y)))]$$

这一部分损失函数的定义与GAN无异。另一部分为循环一致性损失：

$$
\mathcal{L}_{cyc}(G, F) = \mathbb{E}_{x\sim{p_{data}(x)}}[\lVert F(G(x)) - x \rVert _1] + 
\mathbb{E}_{y\sim{p_{data}(y)}}[\lVert G(F(y)) - y \rVert _1]
$$

其中，$\lVert F(G(x)) - x \rVert _1$为$L1$损失函数，目标是最小化真实值和估计值之差绝对值的和，即：

$$L1 = \sum_{n=1}^{N} |x' - x| $$

则CycleGAN整体的损失函数为：


$$
\mathcal{L}(G, F, D_X, D_Y) = \mathcal{L}_{GAN}(G, D_Y, X, Y) 
+ \mathcal{L}_{GAN}(F, D_X, Y, X)
+ \mathcal{L}_{cyc}(G, F) 
$$

**恒等映射损失**

在CycleGAN的实践中，作者引入了一个恒等映射损失函数(identity loss)，即把目标数据空间的样本传给生成器，来训练损失。生成伪造图片时，如果一些内容或特征已经与目标接近，则不应该对图片进行太大的改动。在图片风格转换时，内容是一样的，只是风格不同，这样可以更好的保留原图的一些特征，如下图所示：

![identity-loss-effect](img/identity-loss-effect.png)

以上是论文中作者给出的例子，用于从实景图片到莫奈绘画风格的转换。可以看到，加入恒等映射的损失函数后，生成图像时避免了对原图大范围特征（颜色）的过度修改。

在实践中，identity loss的做法与cycle consistency loss类似。不一样的地方在于，生成模型接受的参数是目标数据空间的样本，用这种样本与原始的目标空间真实数据进行$L1$损失函数的计算。例如$G$本来是映射$X \to {Y}$的关系，但是我们给$G$传入$Y$的样本，计算$G(Y)与Y$的差值作为$L1$损失函数，因此identity loss的定义为：

$$
\mathcal{L}_{identity}(G, F) = \mathbb{E}_{x\sim{p_{data}(x)}}[\lVert F(x) - x \rVert _1] + 
\mathbb{E}_{y\sim{p_{data}(y)}}[\lVert G(y) - y \rVert _1]
$$

identity loss的思路引用于Yaniv Taigman等人的研究，具体原理和数学推演可以参考论文[《UNSUPERVISED CROSS-DOMAIN IMAGE GENERATION》](https://arxiv.org/pdf/1611.02200.pdf)。

### 实现CycleGAN

下面，我们实现一个简单的CycleGAN结构，并用于物体转换（苹果和橘子的转换）。首先，准备数据和工具代码，由于我们安装了新的库keras_contrib，执行完下面cell后需要重启一下jupyter kernel：

In [ ]:
import os

if os.path.exists('./data') == False:
    from modelarts.session import Session
    sess = Session()
    sess.download_data(bucket_path='ai-course-common-26/GAN/gan.tar.gz',
                       path='./gan.tgz')
    ! tar zxf gan.tgz
    ! cd data/keras-contrib-master/ && python setup.py install

接下来，定义generator和discriminator的构造方法：

In [ ]:
from keras.layers import Conv2D, LeakyReLU, UpSampling2D, Dropout, Concatenate, Input
from keras_contrib.layers.normalization.instancenormalization import InstanceNormalization
from keras.models import Model

img_rows = 128
img_cols = 128
channels = 3
img_shape = (img_rows, img_cols, channels)

patch = int(img_rows / 2**4)
disc_patch = (patch, patch, 1)

# Number of filters in the first layer of G and D
gf = 32
df = 64

# Loss weights
lambda_cycle = 10.0               # Cycle-consistency loss
lambda_id = 0.1 * lambda_cycle    # Identity loss


def build_generator():
    def conv2d(layer_input, filters, f_size=4):
        """Layers used during downsampling"""
        d = Conv2D(filters, kernel_size=f_size, strides=2, padding='same')(layer_input)
        d = LeakyReLU(alpha=0.2)(d)
        d = InstanceNormalization()(d)
        return d

    def deconv2d(layer_input, skip_input, filters, f_size=4, dropout_rate=0):
        """Layers used during upsampling"""
        u = UpSampling2D(size=2)(layer_input)
        u = Conv2D(filters, kernel_size=f_size, strides=1, padding='same', activation='relu')(u)
        if dropout_rate:
            u = Dropout(dropout_rate)(u)
        u = InstanceNormalization()(u)
        u = Concatenate()([u, skip_input])
        return u

    # Image input
    d0 = Input(shape=img_shape)

    # Downsampling
    d1 = conv2d(d0, gf)
    d2 = conv2d(d1, gf*2)
    d3 = conv2d(d2, gf*4)
    d4 = conv2d(d3, gf*8)

    # Upsampling
    u1 = deconv2d(d4, d3, gf*4)
    u2 = deconv2d(u1, d2, gf*2)
    u3 = deconv2d(u2, d1, gf)

    u4 = UpSampling2D(size=2)(u3)
    output_img = Conv2D(channels, kernel_size=4, strides=1, padding='same', activation='tanh')(u4)

    return Model(d0, output_img)

def build_discriminator():
    def d_layer(layer_input, filters, f_size=4, normalization=True):
        """Discriminator layer"""
        d = Conv2D(filters, kernel_size=f_size, strides=2, padding='same')(layer_input)
        d = LeakyReLU(alpha=0.2)(d)
        if normalization:
            d = InstanceNormalization()(d)
        return d

    img = Input(shape=img_shape)

    d1 = d_layer(img, df, normalization=False)
    d2 = d_layer(d1, df*2)
    d3 = d_layer(d2, df*4)
    d4 = d_layer(d3, df*8)

    validity = Conv2D(1, kernel_size=4, strides=1, padding='same')(d4)

    return Model(img, validity)

然后，构建生成模型$G，F$和对应的判别模型$D_Y, D_X$：

In [ ]:
from keras.optimizers import Adam

optimizer = Adam(0.0002, 0.5)

# 构建判别模型Dy和Dx
Dx = build_discriminator()
Dy = build_discriminator()

Dx.compile(loss='mse',
    optimizer=optimizer,
    metrics=['accuracy'])
Dy.compile(loss='mse',
    optimizer=optimizer,
    metrics=['accuracy'])

# 构建生成模型G和F
G = build_generator()
F = build_generator()

接下来，我们定义整个CycleGAN的计算图关系。在训练GAN时，我们用均方误差(Mean Square Error)损失函数来替代论文中的$log$似然损失，根据Xudong Mao等人的[论文](https://arxiv.org/pdf/1611.04076.pdf)，使用L2 loss（也称Least Square Errors）比典型的$log$似然更加高效。L2损失的定义为真实值和预测值差的平方和：

$$ LS = \sum_{i=1}^{n} (G(x) - x)^{2} $$

而MSE为对平方和再求平均数：

$$ MSE =  \frac{1}{N}  \sum_{i=1}^{n} (G(x) - x)^{2} $$

在我们使用的框架Keras中，并没有直接提供L2损失函数，而是提供了MSE，常数在求解时候可以忽略，因此不影响GAN的训练。

同样，对cycle consistency loss和identity loss，Keras也没有直接提供L1损失函数，我们也使用MAE(Mean Average Error)替代。

In [ ]:
# 数据空间X和Y的输入
img_X = Input(shape=img_shape)
img_Y = Input(shape=img_shape)

# 生成伪造的Y和X
fake_Y = G(img_X)
fake_X = F(img_Y)
# 将伪造的数据再映射回原来的数据空间
reconstr_X = F(fake_Y)
reconstr_Y = G(fake_X)

# 构建恒等映射，让生成模型在生成数据时不会过度改动原图特征
img_X_id = F(img_X)
img_Y_id = G(img_Y)

# 冻结判别模型，在combined模型中，只训练生成模型
Dx.trainable = False
Dy.trainable = False

# 判别模型对伪造数据进行判定，的判别结果 Discriminators determines validity of translated images
valid_X = Dx(fake_X)
valid_Y = Dy(fake_Y)

# 生成模型在combined model中被训练。
combined = Model(inputs=[img_X, img_Y],
                 outputs=[valid_X, valid_Y,
                          reconstr_X, reconstr_Y,
                          img_X_id, img_Y_id])

# 我们使用MSE损失函数代替GAN论文中的log似然
# 用MAE损失函数代替循环一致性的L1损失函数
combined.compile(loss=['mse', 'mse',
                       'mae', 'mae',
                       'mae', 'mae'],
                 loss_weights=[1, 1,
                               lambda_cycle, lambda_cycle,
                               lambda_id, lambda_id ],
                 optimizer=optimizer)

### 训练模型

训练之前，我们先定义一个助手函数，用来随机生成数据，观察模型效果

In [ ]:
from src.cyclegan.data_loader import DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

# plt.figure(figsize=(20, 10));

def sample_train_results():
    dataset_name = 'apple2orange'
    dataloader = DataLoader(dataset_name=dataset_name,
                                  img_res=(img_rows, img_cols))

    r, c = 2, 3

    imgs_X = dataloader.load_data(domain="A", batch_size=1, is_testing=True)
    imgs_Y = dataloader.load_data(domain="B", batch_size=1, is_testing=True)


    # Translate images to the other domain
    # 图片转换
    fake_Y = G.predict(imgs_X)
    fake_X = F.predict(imgs_Y)

    # 重新将图片转换回原来的数据空间
    reconstr_X = F.predict(fake_Y)
    reconstr_Y = G.predict(fake_X)

    gen_imgs = np.concatenate([imgs_X, fake_Y, reconstr_X, imgs_Y, fake_X, reconstr_Y])

    # Rescale images 0 - 1
    gen_imgs = 0.5 * gen_imgs + 0.5

    titles = ['Original', 'Translated', 'Reconstructed']

    fig, axs = plt.subplots(r, c)

    cnt = 0
    for i in range(r):
        for j in range(c):
            axs[i,j].imshow(gen_imgs[cnt]);
            axs[i, j].set_title(titles[j])
            axs[i,j].axis('off')
            cnt += 1
    
    plt.show();   

下面，我们训练模型，并定期显示模型训练的效果

In [ ]:
import numpy as np

epochs = 3
batch_size = 1
sample_results_interval = 200 # 每训练100个batch观察一次生成模型的结果

valid = np.ones((batch_size,) + disc_patch)
fake = np.zeros((batch_size,) + disc_patch)

dataset_name = 'apple2orange'
dataloader = DataLoader(dataset_name=dataset_name,
                              img_res=(img_rows, img_cols))

import datetime

start_time = datetime.datetime.now()

for epoch in range(epochs):
    for batch_i, (imgs_X, imgs_Y) in enumerate(dataloader.load_batch(batch_size)):
        # 训练判别模型d_X, d_Y

        
        # 首先，生成伪造的Y'=G(x)和伪造的X'=F(y)
        fake_Y = G.predict(imgs_X)
        fake_X = F.predict(imgs_Y)

        # 用真实数据和伪造数据训练判别模型
        Dx_loss_real = Dx.train_on_batch(imgs_X, valid)
        Dx_loss_fake = Dy.train_on_batch(fake_X, fake)
        Dx_loss = 0.5 * np.add(Dx_loss_real, Dx_loss_fake)

        Dy_loss_real = Dy.train_on_batch(imgs_Y, valid)
        Dy_loss_fake = Dy.train_on_batch(fake_Y, fake)
        Dy_loss = 0.5 * np.add(Dy_loss_real, Dy_loss_fake)

        # 对Dx和Dx损失求平均，作为生成模型的整体损失
        d_loss = 0.5 * np.add(Dx_loss, Dy_loss)

        # 在combined model中训练生成模型
        g_loss = combined.train_on_batch([imgs_X, imgs_Y],
                                         [valid, valid,
                                          imgs_X, imgs_Y,
                                          imgs_X, imgs_Y])

        elapsed_time = datetime.datetime.now() - start_time
        
        if batch_i % sample_results_interval == 0:
            print ("[Epoch %d/%d] [Batch %d/%d] [D loss: %f, acc: %3d%%] [G loss: %05f, adv: %05f, recon: %05f, id: %05f] time: %s " \
                                                                    % ( epoch+1, epochs,
                                                                        batch_i, dataloader.n_batches,
                                                                        d_loss[0], 100*d_loss[1],
                                                                        g_loss[0],
                                                                        np.mean(g_loss[1:3]),
                                                                        np.mean(g_loss[3:5]),
                                                                        np.mean(g_loss[5:6]),
                                                                        elapsed_time))
            sample_train_results()

### CycleGAN训练总结


可以看到，刚开始训练时，生成的图片包含较多的噪点，很容易判别。而经过若干论训练，整个CycleGAN已经学习到苹果和橘子的主要特征，包括颜色、纹理，其中视觉上最明显的特征是颜色。在我们的样本中，苹果多为红色，而橘子多为橙色，在转换时，颜色特征的转换非常明显，在一些训练情况较好权重下，从橘子转换为苹果时，也可以看到生成的水果纹理比原图更光滑。


此外，在训练过程中，我们还发现了一些有趣的现象。在一次训练中，橘子样本图片的内容为案板上摆放的橘子和紫色的茄子，模型在训练时将紫色的茄子当成橘子的特征并训练得到了较高的权重，因此后面若干轮的训练中，都把紫色作为一个主要特征，应用到图片的生成和双向重建中，并一直沿着这个优化的方向延续到训练结束。对于这种情况，我们应该适当的提前终止训练，并剔除掉数据集中特征分散、不明确的样本。


![cyclegan-bad-training](img/cyclegan-bad-training.png)
> 训练过程中，遇到特征分散的样本并且模型恰好按照错误的特征进行训练



由于CycleGAN中的模型较多，模型间关系也比较繁杂，模型之间梯度下降的方向不易协调，因此很难用一个指标去衡量整个CycleGAN的模型性能。CycleGAN的作者在公布的Github项目说明中也阐述了这一点，并针对CycleGAN不易训练的特点给出了[若干提示](https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix/blob/master/docs/tips.md)，建议开发者周期性的用训练好的参数去生成数据并观察结果。此外，行业中也有一些相关的优化技巧的讨论，如在生成伪造数据时加入高斯分布以避免判别模型捕捉到生成图像颜色值的数字特征，以及[label smoothing](https://arxiv.org/abs/1906.02629)技术等。


- **扩展1**：参考CycleGAN作者给出的训练建议，以及业界相关的优化技巧，优化CycleGAN的训练
- **扩展2**：阅读src/cyclegan/目录下的相关代码和脚本，尝试下载其他数据集训练CycleGAN并观察模型效果